# Benchmark Plan — Batch Read of Patch Data for Model Training (TileDB)

## Overview

This benchmark measures how quickly a TileDB-backed dataloader can retrieve a
random batch of 10 000 patches from an array of 1 000 000 patches, simulating
the random-access read pattern used during DL model training.

## Benchmark Plan

- **Operation measured** — `DenseArray.multi_index[sorted_patch_ids]`: a
  single call that reads 10 000 non-contiguous patches by their `patch_id`
  index and returns them as a `(10000, 32, 32, 3)` uint8 ndarray.

- **Table / array size** — 1 000 000 patches, each `(32, 32, 3)` uint8
  (~3 GB uncompressed; LZ4-compressed on disk).  This matches the first
  benchmark row (Zarr baseline) for a direct comparison.

- **Index / data distribution** — `patch_id` dimension spans 0–999 999;
  the 10 000 query IDs are drawn uniformly at random (seeded for
  reproducibility) and *sorted* before the query to improve IO locality
  across TileDB tiles.

- **Environment setup** — A 4-D TileDB dense array is created idempotently
  on the local filesystem (`/tmp`).  Data is seeded in batches of 10 000 to
  cap peak RAM usage (~300 MB per batch).  A single warm-up read is issued
  to populate OS and TileDB caches before timing starts.

- **Timing method** — `time.perf_counter()` wraps **only** the
  `multi_index` call.  Setup, seeding, warm-up, and cleanup are all
  **outside** the timed block.

- **Benchmark repetition** — The timed read is executed 3 times against the
  same on-disk array; the **median** elapsed time is reported.

- **Edge cases** —
  - Non-contiguous patch IDs trigger multi-range tile reads; IDs are sorted
    to minimise tile re-reads.
  - `try/finally` guarantees the TileDB array is removed even on failure.
  - TileDB's LZ4-compressed tiles mean decompression cost is included in the
    measured time (realistic).

## Comparison to Zarr Baseline

The existing Zarr benchmark (row 0) uses NumPy advanced indexing
(`z[sorted_ids]`) on the same array shape and size.  TileDB uses its native
`multi_index` API, which constructs an equivalent multi-range subarray query.


In [1]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import os
import time

import numpy as np
import tiledb

print(f"TileDB version : {tiledb.__version__}")
print(f"NumPy  version : {np.__version__}")

# ── Config ────────────────────────────────────────────────────────────────────
TILEDB_PATH   = "/tmp/tiledb_patch_benchmark_1M"
NUM_PATCHES   = 1_000_000
PATCH_H       = 32
PATCH_W       = 32
PATCH_C       = 3
BATCH_SIZE    = 10_000
SEED_BATCH    = 10_000
TILE_PATCH_ID = 1024
N_TRIALS      = 3

print(f"\nArray path     : {TILEDB_PATH}")
print(f"Total patches  : {NUM_PATCHES:,}")
print(f"Patch shape    : ({PATCH_H}, {PATCH_W}, {PATCH_C})")
estimated_gb = NUM_PATCHES * PATCH_H * PATCH_W * PATCH_C / 1e9
print(f"Uncompressed   : ~{estimated_gb:.1f} GB")

TileDB version : 0.36.1
NumPy  version : 2.2.6

Array path     : /tmp/tiledb_patch_benchmark_1M
Total patches  : 1,000,000
Patch shape    : (32, 32, 3)
Uncompressed   : ~3.1 GB


In [2]:
# ── Array creation and seeding ────────────────────────────────────────────────
try:
    # Idempotent teardown
    if tiledb.array_exists(TILEDB_PATH):
        tiledb.remove(TILEDB_PATH)

    # Schema
    domain = tiledb.Domain(
        tiledb.Dim(
            name="patch_id",
            domain=(0, NUM_PATCHES - 1),
            tile=TILE_PATCH_ID,
            dtype=np.int64,
        ),
        tiledb.Dim(name="h", domain=(0, PATCH_H - 1), tile=PATCH_H, dtype=np.int64),
        tiledb.Dim(name="w", domain=(0, PATCH_W - 1), tile=PATCH_W, dtype=np.int64),
        tiledb.Dim(name="c", domain=(0, PATCH_C - 1), tile=PATCH_C, dtype=np.int64),
    )
    pixel_attr = tiledb.Attr(
        name="pixel",
        dtype=np.uint8,
        var=False,
        filters=tiledb.FilterList([tiledb.LZ4Filter()]),
    )
    schema = tiledb.ArraySchema(
        domain=domain,
        attrs=[pixel_attr],
        sparse=False,
        cell_order="row-major",
        tile_order="row-major",
    )
    print("Creating TileDB dense array...")
    tiledb.DenseArray.create(TILEDB_PATH, schema)

    # Seed in batches
    rng = np.random.default_rng(2024)
    t0 = time.perf_counter()
    print(f"Seeding {NUM_PATCHES:,} patches in batches of {SEED_BATCH:,}...")
    for batch_start in range(0, NUM_PATCHES, SEED_BATCH):
        batch_end = min(batch_start + SEED_BATCH, NUM_PATCHES)
        chunk = rng.integers(
            0, 256,
            size=(batch_end - batch_start, PATCH_H, PATCH_W, PATCH_C),
            dtype=np.uint8,
        )
        with tiledb.DenseArray(TILEDB_PATH, mode="w") as A:
            A[batch_start:batch_end] = {"pixel": chunk}
        if batch_start % 100_000 == 0:
            print(f"  {batch_start:>9,} / {NUM_PATCHES:,}  ({time.perf_counter()-t0:.1f}s elapsed)")
    print(f"Seeding complete — {time.perf_counter()-t0:.1f}s total")

    # Quick sanity check
    with tiledb.DenseArray(TILEDB_PATH, mode="r") as A:
        sample = A[0:3]["pixel"]
        assert sample.shape == (3, PATCH_H, PATCH_W, PATCH_C)
    print("Array is ready for benchmarking.")

except Exception:
    if tiledb.array_exists(TILEDB_PATH):
        tiledb.remove(TILEDB_PATH)
    raise

Creating TileDB dense array...
Seeding 1,000,000 patches in batches of 10,000...
          0 / 1,000,000  (0.3s elapsed)
    100,000 / 1,000,000  (1.9s elapsed)
    200,000 / 1,000,000  (3.5s elapsed)
    300,000 / 1,000,000  (5.0s elapsed)
    400,000 / 1,000,000  (7.0s elapsed)
    500,000 / 1,000,000  (8.5s elapsed)
    600,000 / 1,000,000  (9.6s elapsed)
    700,000 / 1,000,000  (10.7s elapsed)
    800,000 / 1,000,000  (12.6s elapsed)
    900,000 / 1,000,000  (13.7s elapsed)
Seeding complete — 14.6s total
Array is ready for benchmarking.


In [3]:
# ── Benchmark: timed batch read ───────────────────────────────────────────────
try:
    # Sample 10k random patch IDs (sorted for IO locality)
    rng2 = np.random.default_rng(42)
    patch_ids = np.sort(
        rng2.choice(NUM_PATCHES, size=BATCH_SIZE, replace=False)
    ).tolist()

    # Warm-up: single patch read to populate OS/TileDB caches
    with tiledb.DenseArray(TILEDB_PATH, mode="r") as A:
        _ = A[0:1]["pixel"]
    print("Warm-up read done.")

    # Timed trials
    elapsed_list = []
    print(f"\n--- Timed trials (batch_size={BATCH_SIZE:,}) ---")
    for trial in range(1, N_TRIALS + 1):
        with tiledb.DenseArray(TILEDB_PATH, mode="r") as A:
            t_start = time.perf_counter()
            result  = A.multi_index[patch_ids]
            patches = result["pixel"]
            elapsed = time.perf_counter() - t_start
        elapsed_list.append(elapsed)
        throughput = BATCH_SIZE / elapsed
        print(
            f"Trial {trial}: elapsed={elapsed:.3f}s  "
            f"throughput={throughput:,.0f} r/s  "
            f"shape={patches.shape}"
        )

    best_elapsed   = min(elapsed_list)
    median_elapsed = sorted(elapsed_list)[len(elapsed_list) // 2]
    mean_elapsed   = sum(elapsed_list) / len(elapsed_list)

    print(f"\nBest   : {best_elapsed:.3f}s  ({BATCH_SIZE/best_elapsed:,.0f} r/s)")
    print(f"Median : {median_elapsed:.3f}s  ({BATCH_SIZE/median_elapsed:,.0f} r/s)")
    print(f"Mean   : {mean_elapsed:.3f}s  ({BATCH_SIZE/mean_elapsed:,.0f} r/s)")
    print(
        f"\nFINAL RESULT (median): "
        f"Elapsed={median_elapsed:.3f}s, "
        f"Throughput={BATCH_SIZE/median_elapsed:,.0f} r/s"
    )

finally:
    # Cleanup
    if tiledb.array_exists(TILEDB_PATH):
        tiledb.remove(TILEDB_PATH)
        print("\nTileDB array cleaned up.")

Warm-up read done.

--- Timed trials (batch_size=10,000) ---
Trial 1: elapsed=0.763s  throughput=13,109 r/s  shape=(10000, 32, 32, 3)
Trial 2: elapsed=0.680s  throughput=14,699 r/s  shape=(10000, 32, 32, 3)
Trial 3: elapsed=0.660s  throughput=15,149 r/s  shape=(10000, 32, 32, 3)

Best   : 0.660s  (15,149 r/s)
Median : 0.680s  (14,699 r/s)
Mean   : 0.701s  (14,286 r/s)

FINAL RESULT (median): Elapsed=0.680s, Throughput=14,699 r/s

TileDB array cleaned up.


# Result Summary

## Execution Output

```
TileDB version : 0.36.1
NumPy  version : 2.2.6

Creating TileDB dense array...
Seeding 1,000,000 patches in batches of 10,000...
Seeding complete

Warm-up read done.

--- Timed trials (batch_size=10,000) ---
Trial 1: elapsed=0.763s  throughput=13,109 r/s  shape=(10000, 32, 32, 3)
Trial 2: elapsed=0.680s  throughput=14,699 r/s  shape=(10000, 32, 32, 3)
Trial 3: elapsed=0.660s  throughput=15,149 r/s  shape=(10000, 32, 32, 3)

Best   : 0.660s  (15,149 r/s)
Median : 0.680s  (14,699 r/s)
Mean   : 0.701s  (14,286 r/s)

FINAL RESULT (median): Elapsed=0.680s, Throughput=14,699 r/s

TileDB array cleaned up.
```

## Key Metrics

| Metric             | Value             |
| ------------------ | ----------------- |
| Array backend      | TileDB 0.36.1     |
| Array size         | 1 000 000 patches |
| Batch size         | 10 000 patches    |
| Patch shape        | (32, 32, 3) uint8 |
| Read API           | `multi_index`     |
| Compression        | LZ4               |
| **Elapsed (median)** | **0.680 s**     |
| **Throughput (median)** | **14,699 r/s** |
| Best elapsed       | 0.660 s           |
| Best throughput    | 15,149 r/s        |

## CSV Result String

```
"0.680s, ~14,699 r/s"
```

## Notes

- **TileDB vs Zarr**: Both backends achieve comparable throughput on this
  workload (~14–15 k patches/s).  TileDB's `multi_index` API issues a
  single multi-range subarray query, while the Zarr baseline uses NumPy
  advanced indexing.

- **Read amplification**: Random patch IDs span all 1 000 tiles along the
  `patch_id` dimension (tile size 1024).  TileDB must decompress entire
  tiles to extract individual rows; sorting the IDs minimises repeated tile
  decompression.

- **Training suitability**: At ~14 700 r/s, a DataLoader with a single
  worker can supply a batch of 10 000 patches in under 0.7 s, fast enough
  for most training loops on moderate GPU hardware.

- **Scalability**: All three trials are within 15 % of each other, indicating
  stable IO behaviour.  Larger arrays would require re-running to verify
  that performance does not degrade (TileDB fragments accumulate; periodic
  consolidation is recommended).

- **Infrastructure**: Benchmark ran entirely on local disk (`/tmp`);
  no PostgreSQL connection was required.
